# Backtest Synthetic

## Actual-Data Calibration

- **Purpose:** Calibrate the synthetic mean-reverting return experiment from observed strategy behavior.
- **Settings:** `ar1_clip=(0.05, 0.99)`; the observed mean sets the forecast, residual standard deviation sets shocks, and absolute-return quantiles `0.50` and `0.75` set barriers.
- **Data:** Estimate every calibration input from development meta-filtered net returns.
- **Decision:** Exclude holdout outcomes from calibration.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.backtesting.backtest_synthetic import synthetic_trading_rule_experiment

result_dir = PROJECT_ROOT / "data/backtest_results"
event_returns = pd.read_parquet(result_dir / "event_strategy_returns.parquet").sort_index()
observed = event_returns.loc[event_returns["partition"].eq("development"), "meta_filtered_net_return"].astype(float)

lagged = observed.iloc[:-1].to_numpy()
current = observed.iloc[1:].to_numpy()
phi = float(np.polyfit(lagged, current, deg=1)[0])
phi = float(np.clip(phi, 0.05, 0.99))
half_life = float(-np.log(2.0) / np.log(phi))
forecast = float(observed.mean())
residuals = current - (forecast * (1.0 - phi) + phi * lagged)
sigma = float(np.std(residuals, ddof=1))
absolute_quantiles = observed.abs().quantile([0.50, 0.75]).clip(lower=1e-6)

calibration = pd.Series(
    {
        "forecast": forecast,
        "phi": phi,
        "half_life": half_life,
        "sigma": sigma,
        "median_absolute_return": float(absolute_quantiles.loc[0.50]),
        "upper_quartile_absolute_return": float(absolute_quantiles.loc[0.75]),
        "observations": len(observed),
        "random_state": 42,
    },
    name="value",
)
display(calibration.to_frame())


,value
forecast,0.000600
phi,0.129104
half_life,0.338593
sigma,0.004227
median_absolute_return,0.001017
upper_quartile_absolute_return,0.002149
observations,176.000000
random_state,42.000000


## Seeded Synthetic Trading-Rule Experiment

- **Purpose:** Run a reproducible synthetic trading-rule sensitivity experiment with the calibrated process.
- **Settings:** `iterations=2000`; `max_holding_period=50` observations; `random_state=42`.
- **Data:** Generate simulated trading-rule outcomes from the development-calibrated process.
- **Decision:** Use the output only as a sensitivity diagnostic and never to replace or retune the observed primary and meta models.

In [2]:
simulation = synthetic_trading_rule_experiment(
    forecasts=[forecast],
    half_lives=[half_life],
    profit_taking_range=absolute_quantiles.to_numpy(),
    stop_loss_range=absolute_quantiles.to_numpy(),
    sigma=sigma,
    num_iterations=2_000,
    max_holding_period=50,
    random_state=42,
)

display(simulation.sort_values("sharpe_ratio", ascending=False))


,forecast,half_life,sigma,max_holding_period,profit_taking,stop_loss,mean,std,sharpe_ratio
1,0.0006,0.338593,0.004227,50,0.001017,0.002149,0.000679,0.004918,0.138096
3,0.0006,0.338593,0.004227,50,0.002149,0.002149,0.000578,0.005321,0.108604
0,0.0006,0.338593,0.004227,50,0.001017,0.001017,0.000441,0.004698,0.093968
2,0.0006,0.338593,0.004227,50,0.002149,0.001017,0.000314,0.005014,0.062614


## Results, Limitations, and Handoff

- **Purpose:** Summarize and hand off the seeded, actual-data-calibrated simulation results.
- **Settings:** No new analytical parameters; results inherit the calibrated OU approximation and one-year development sample.
- **Data:** Summarize the simulated Sharpe-ratio results from the calibrated process.
- **Decision:** Treat synthetic Sharpe ratios as sensitivity outputs rather than profitability evidence.